In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

# Each notebook has its own kernel, so load the raw data again here
PROJECT_ROOT = Path.cwd().parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "superstore.csv"

# The Superstore CSV is Windows-1252 encoded, not UTF-8
df = pd.read_csv(RAW_FILE, encoding="cp1252")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
df_clean = df.copy()

print("Original shape:", df.shape)
print("Cleaning shape:", df_clean.shape)

Original shape: (9994, 21)
Cleaning shape: (9994, 21)


In [ ]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df_clean.columns.tolist()

In [5]:
df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    errors="coerce"
)

df_clean["ship_date"] = pd.to_datetime(
    df_clean["ship_date"],
    errors="coerce"
)

In [6]:
print(df_clean["order_date"].dtype)
print(df_clean["ship_date"].dtype)

datetime64[us]
datetime64[us]


In [7]:
print("Earliest order:", df_clean["order_date"].min())
print("Latest order:", df_clean["order_date"].max())

print("Earliest shipment:", df_clean["ship_date"].min())
print("Latest shipment:", df_clean["ship_date"].max())

Earliest order: 2014-01-03 00:00:00
Latest order: 2017-12-30 00:00:00
Earliest shipment: 2014-01-07 00:00:00
Latest shipment: 2018-01-05 00:00:00


In [8]:
invalid_shipments = df_clean[
    df_clean["ship_date"] < df_clean["order_date"]
]

print("Shipments before order:", len(invalid_shipments))

Shipments before order: 0


In [9]:
df_clean[
    ["sales", "quantity", "discount", "profit"]
].dtypes

sales       float64
quantity      int64
discount    float64
profit      float64
dtype: object

In [10]:
numeric_columns = [
    "sales",
    "quantity",
    "discount",
    "profit"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

In [11]:
df_clean[numeric_columns].isna().sum()

sales       0
quantity    0
discount    0
profit      0
dtype: int64

In [12]:
print("Quantity <= 0:",
      (df_clean["quantity"] <= 0).sum())

Quantity <= 0: 0


In [13]:
print("Sales < 0:",
      (df_clean["sales"] < 0).sum())

Sales < 0: 0


In [14]:
print("Discount < 0:",
      (df_clean["discount"] < 0).sum())

print("Discount > 100%:",
      (df_clean["discount"] > 1).sum())

Discount < 0: 0
Discount > 100%: 0


In [15]:
print("Negative profit:",
      (df_clean["profit"] < 0).sum())

Negative profit: 1871


In [18]:
# Make sure hyphenated names (e.g. "sub-category") use underscores
df_clean.columns = df_clean.columns.str.replace("-", "_")

text_columns = [
    "order_id",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name"
]

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

In [19]:
print(df_clean["category"].unique())

<StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str


In [20]:
print(df_clean["region"].unique())

<StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str


In [21]:
print(
    "Exact duplicate rows:",
    df_clean.duplicated().sum()
)

Exact duplicate rows: 0


In [22]:
print(
    "Duplicate row IDs:",
    df_clean["row_id"].duplicated().sum()
)

Duplicate row IDs: 0


In [23]:
print("Rows:", len(df_clean))
print("Unique row IDs:", df_clean["row_id"].nunique())
print("Unique orders:", df_clean["order_id"].nunique())
print("Unique customers:", df_clean["customer_id"].nunique())
print("Unique products:", df_clean["product_id"].nunique())

Rows: 9994
Unique row IDs: 9994
Unique orders: 5009
Unique customers: 793
Unique products: 1862


In [24]:
df_clean["revenue"] = df_clean["sales"]

df_clean["cogs"] = (
    df_clean["sales"] -
    df_clean["profit"]
)

In [25]:
df_clean["profit_margin"] = np.where(
    df_clean["revenue"] != 0,
    (df_clean["profit"] / df_clean["revenue"]) * 100,
    np.nan
)

In [26]:
profit_check = (
    df_clean["revenue"]
    - df_clean["cogs"]
)

difference = (
    profit_check -
    df_clean["profit"]
).abs()

print("Maximum calculation difference:",
      difference.max())

Maximum calculation difference: 1.5916157281026244e-12


In [27]:
df_clean["year"] = df_clean["order_date"].dt.year

df_clean["month"] = df_clean["order_date"].dt.month

df_clean["month_name"] = (
    df_clean["order_date"].dt.month_name()
)

df_clean["quarter"] = (
    df_clean["order_date"].dt.quarter
)

In [28]:
df_clean["month_start"] = (
    df_clean["order_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [29]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 29 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   row_id         9994 non-null   int64         
 1   order_id       9994 non-null   str           
 2   order_date     9994 non-null   datetime64[us]
 3   ship_date      9994 non-null   datetime64[us]
 4   ship_mode      9994 non-null   str           
 5   customer_id    9994 non-null   str           
 6   customer_name  9994 non-null   str           
 7   segment        9994 non-null   str           
 8   country        9994 non-null   str           
 9   city           9994 non-null   str           
 10  state          9994 non-null   str           
 11  postal_code    9994 non-null   int64         
 12  region         9994 non-null   str           
 13  product_id     9994 non-null   str           
 14  category       9994 non-null   str           
 15  sub_category   9994 non-null   s

In [30]:
print("Final rows:", len(df_clean))
print("Final columns:", len(df_clean.columns))

Final rows: 9994
Final columns: 29


In [31]:
print("========== DATA QUALITY CHECK ==========")

print(
    "Rows:",
    len(df_clean)
)

print(
    "Columns:",
    len(df_clean.columns)
)

print(
    "Missing values:",
    df_clean.isna().sum().sum()
)

print(
    "Exact duplicates:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate Row IDs:",
    df_clean["row_id"].duplicated().sum()
)

print(
    "Invalid shipment dates:",
    (
        df_clean["ship_date"] <
        df_clean["order_date"]
    ).sum()
)

print(
    "Negative sales:",
    (df_clean["sales"] < 0).sum()
)

print(
    "Negative quantity:",
    (df_clean["quantity"] < 0).sum()
)

print(
    "Negative profit:",
    (df_clean["profit"] < 0).sum()
)

print("========================================")

========== DATA QUALITY CHECK ==========
Rows: 9994
Columns: 29
Missing values: 0
Exact duplicates: 0
Duplicate Row IDs: 0
Invalid shipment dates: 0
Negative sales: 0
Negative quantity: 0
Negative profit: 1871


In [32]:
PROCESSED_FILE = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "superstore_clean.csv"
)

df_clean.to_csv(
    PROCESSED_FILE,
    index=False
)

print("Clean dataset saved to:")
print(PROCESSED_FILE)

Clean dataset saved to:
c:\Users\Aniket Mane\Desktop\CEO-Dashboard\data\processed\superstore_clean.csv
